<a href="https://colab.research.google.com/github/Karandeep09/transformer/blob/main/MaskedSelfAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import torch
import torch.nn as nn
import math

class MaskedSelfAttention(nn.Module):
  def __init__(self, embed_size):
    super().__init__()
    self.embed_size = embed_size
    self.q_w = nn.Linear(in_features=embed_size, out_features=embed_size, bias=False)
    self.k_w = nn.Linear(in_features=embed_size, out_features=embed_size, bias=False)
    self.v_w = nn.Linear(in_features=embed_size, out_features=embed_size, bias=False)
  def forward(self, embeddings, mask=None):
    q = self.q_w(embeddings)
    k = self.k_w(embeddings)
    v = self.v_w(embeddings)
    sims = q @ k.transpose(-1, -2)
    scaled_sims = sims / math.sqrt(k.size(-1))
    if mask is not None:
      scaled_sims = scaled_sims.masked_fill(mask = mask, value = -1e9)
    attention_percents = torch.softmax(scaled_sims, dim=-1)
    print("Attention Percents = ", attention_percents)
    return attention_percents @ v


selfAttention = MaskedSelfAttention(2)
token_embeddings = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
tokens = token_embeddings.size(-2)
mask = torch.tril(torch.ones(tokens, tokens))
mask = mask == 0
print("Mask = ", mask)
attention_scores = selfAttention(token_embeddings, mask)
print("Masked Self Attention Scores = ", attention_scores)

Mask =  tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
Attention Percents =  tensor([[1.0000, 0.0000, 0.0000],
        [0.4474, 0.5526, 0.0000],
        [0.2211, 0.3189, 0.4600]], grad_fn=<SoftmaxBackward0>)
Masked Self Attention Scores =  tensor([[ 1.3216, -0.5330],
        [ 2.0595, -0.6222],
        [ 2.9759, -0.7331]], grad_fn=<MmBackward0>)


In [4]:
import math

# 1. Print the transposed weight matrices of the linear layers
print("q_w transpose:", selfAttention.q_w.weight.transpose(-2, -1))
print("k_w transpose:", selfAttention.k_w.weight.transpose(-2, -1))
print("v_w transpose:", selfAttention.v_w.weight.transpose(-2, -1))

# 2. Perform manual operations
q = token_embeddings @ selfAttention.q_w.weight.transpose(-2, -1)
k = token_embeddings @ selfAttention.k_w.weight.transpose(-2, -1)
v = token_embeddings @ selfAttention.v_w.weight.transpose(-2, -1)

sims = q @ k.transpose(-2, -1)
scaled_sims = sims / math.sqrt(k.size(-1))

# Apply the same mask logic
manual_scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)
manual_attention_percents = torch.softmax(manual_scaled_sims, dim=-1)
manual_output = manual_attention_percents @ v

# 3. Compare with the module output
print("Attention Scores:", attention_scores)
print("Manual Output:", manual_output)
print("Matches:", torch.allclose(attention_scores, manual_output))

q_w transpose: tensor([[ 0.1954, -0.2151],
        [ 0.4343,  0.2653]], grad_fn=<TransposeBackward0>)
k_w transpose: tensor([[ 0.5564,  0.3391],
        [ 0.1130, -0.2184]], grad_fn=<TransposeBackward0>)
v_w transpose: tensor([[ 0.2517,  0.3779],
        [-0.6516, -0.0669]], grad_fn=<TransposeBackward0>)
Attention Scores: tensor([[-1.0515,  0.2441],
        [-1.7765,  0.8079],
        [-2.6258,  1.4684]], grad_fn=<MmBackward0>)
Manual Output: tensor([[-1.0515,  0.2441],
        [-1.7765,  0.8079],
        [-2.6258,  1.4684]], grad_fn=<MmBackward0>)
Matches: True
